# Aula 4 — Regressão Linear
## *Ou: como transformar uma correlação em uma previsão*

> **Data:** *(preencher)*  
> **Professor(a):** *(preencher)*

## 🤔 Antes de começar: da correlação para a previsão

Na **Aula 3** vocês aprenderam a *medir* a relação entre duas variáveis. Fizeram heatmaps,
olharam para a correlação e concluíram coisas do tipo: *"essas duas variáveis andam juntas"*.

Só que correlação responde apenas **"andam juntas?"** e **"o quanto?"**.

Ela não responde a pergunta que todo mundo quer fazer:

> *"Beleza, andam juntas. Mas então **quanto** eu devo esperar de preço para um diamante de 1,2 quilate?"*

É aqui que entra a **Regressão Linear**. Em vez de só medir a força da relação, ela **traça uma
reta** que descreve essa relação, e com essa reta você consegue *prever* valores.

```
Aula 3:  "carat e price têm correlação de 0,92"     → mede a relação
Aula 4:  "cada quilate a mais vale ~US$ 7,8 mil"    → quantifica e prevê
```

**É o mesmo fenômeno, mas agora com um número na ponta.**

## 1. Nosso problema de hoje: o preço dos diamantes 💎

Vamos trabalhar com o dataset **diamonds**, que contém informações sobre ~54.000 diamantes.

Hoje vamos usar apenas as colunas **numéricas**, porque regressão linear trabalha com números e
ainda não aprendemos a converter texto em número (isso fica para a Aula 5).

| Coluna | O que é |
|---|---|
| `carat` | Peso do diamante em quilates (tamanho) |
| `depth` | Profundidade percentual |
| `table` | Largura do topo em relação ao ponto mais largo |
| `x`, `y`, `z` | Dimensões físicas em mm |
| `price` | **Preço em dólares (nosso alvo!)** |

**Nosso objetivo:** prever o preço (`price`) a partir das características numéricas.

Vamos carregar os dados:

In [ ]:
# Importando as bibliotecas que vamos usar
import shutil
import urllib.request
from pathlib import Path

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Na primeira vez a base é baixada para data_raw/ (o original, que fica de
# referência) e copiada para data/ (a cópia de trabalho, que é a que vamos usar).
# Bagunçou a base? Apague data/diamonds.csv e rode esta célula de novo.
url = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/diamonds.csv'
Path('data_raw').mkdir(exist_ok=True)
Path('data').mkdir(exist_ok=True)

if not Path('data_raw/diamonds.csv').exists():
    print('Baixando diamonds.csv...')
    urllib.request.urlretrieve(url, 'data_raw/diamonds.csv')
if not Path('data/diamonds.csv').exists():
    shutil.copy('data_raw/diamonds.csv', 'data/diamonds.csv')

df = pd.read_csv('data/diamonds.csv')

# Primeiras linhas
df.head()

### Relembrando a Aula 3: quem anda junto com o preço?

Antes de traçar qualquer reta, vale olhar o **heatmap de correlação**, exatamente como fizemos na
aula passada.

In [ ]:
# Só as colunas numéricas


**O que o heatmap conta:**

* `carat` tem correlação altíssima com `price` (~0,92): diamante maior, preço maior. Sem surpresa.
* `x`, `y`, `z` (as dimensões físicas) também são fortemente correlacionadas com o preço, e
  também **entre si**, porque todas medem "tamanho". Guarde essa informação, ela volta no fim da aula.
* `depth` e `table` quase não se correlacionam com o preço.

Até aqui, nada novo. Vamos olhar a relação mais forte de perto, num scatterplot:

In [ ]:
# A relação mais forte da base: quilate x preço


Olhando esse gráfico, seu cérebro já está fazendo regressão sozinho: você **imagina uma linha**
passando no meio da nuvem de pontos.

A Regressão Linear faz isso de forma matemática, e sem chute.

## 2. Regressão Linear: o modelo mais honesto do mundo

### O que é?

Regressão Linear é um modelo que tenta encontrar uma **linha reta** que melhor representa a relação
entre as variáveis de entrada e o resultado.

### Analogia: previsão de salário

Imagine que você sabe que:

* Com 0 anos de experiência, o salário é R$ 3.000
* Com 5 anos, o salário é R$ 8.000

Regressão linear traçaria uma reta entre esses pontos e usaria ela para prever: *"com 3 anos de
experiência, o salário deve ser uns R$ 6.000"*.

### A fórmula (sem entrar em pânico)

A fórmula básica é:

```
preço = a₁ × quilate + a₂ × profundidade + a₃ × largura + ... + b
```

Onde:

* **a₁, a₂, a₃...** são os **coeficientes** (o peso de cada característica)
* **b** é o **intercepto** (o valor base quando tudo é zero)

O modelo aprende esses valores durante o treinamento.

### E como ele escolhe "a melhor" reta?

Entre todas as retas possíveis, o modelo escolhe aquela que deixa os pontos **o mais perto possível**
dela. Em outras palavras: a reta que **minimiza a soma dos erros ao quadrado**.

É por isso que a regressão linear é honesta: ela não esconde o erro, é literalmente construída para
deixá-lo o menor possível.

## 3. Uma variável, uma reta

Antes de jogar todas as variáveis no modelo, vamos fazer o caso mais simples possível: prever o
preço **usando apenas o quilate**.

Assim dá para *ver* a reta que o modelo encontrou.

In [ ]:
# X precisa ser uma tabela (2 dimensões), mesmo com uma coluna só. Por isso os colchetes duplos.


Leia de novo a última linha que o código imprimiu. Aquilo é uma **frase sobre o mundo real**, não
um número abstrato:

> *"A cada quilate a mais, o preço sobe cerca de US$ 7,8 mil."*

Pronto: a correlação da Aula 3 virou um **número com unidade**, que você pode usar para prever.

Vamos desenhar essa reta em cima dos pontos:

In [ ]:
# A reta ajustada em cima da nuvem de pontos

# Previsão do modelo para cada valor de quilate


> **Repare:** a reta não passa por cima de todos os pontos, e nem deveria. Ela representa a
> **tendência média**. A distância entre cada ponto e a reta é o **erro** daquela previsão, e é
> exatamente isso que vamos medir daqui a pouco.

## 4. Treinando o modelo (agora com todas as variáveis)

Uma variável só é didático, mas joga fora informação. Vamos usar **todas as colunas numéricas** de
uma vez. A lógica é idêntica, só que agora a "reta" existe em várias dimensões.

Antes, precisamos separar:

* **Features (X):** as colunas que usamos como entrada (as características do diamante)
* **Target (y):** a coluna que queremos prever (o preço)

In [ ]:
# Separando features (X) e target (y)


In [ ]:
# Criando o modelo

# Treinando: o modelo aprende a relação entre features e preço


Pronto! Só isso. O scikit-learn cuida de toda a matemática por baixo do capô.

Agora vamos fazer previsões:

## 5. Fazendo Previsões

In [ ]:
# Fazendo previsões

# Comparando as primeiras previsões com os valores reais


Já deu para ter uma ideia! Os valores não são idênticos (isso seria suspeito), mas estão na mesma
faixa. Vamos medir isso direito.

## 6. Avaliando o Modelo

Como saber se o modelo é bom?

Usamos métricas de erro. Vamos usar duas:

### MAE (Mean Absolute Error, ou Erro Médio Absoluto)

Em português: *"em média, o modelo erra por quantos dólares?"*

Exemplo: MAE = 800 significa que, em média, a previsão erra por ±$800.

### R² (R-quadrado)

Mede o quanto o modelo explica a variação dos preços. Vai de 0 a 1:

* 0 = o modelo não explica nada (inútil)
* 1 = o modelo é perfeito (suspeito!)
* 0.85 = o modelo explica 85% da variação nos preços (bom!)

> 💡 **Curiosidade:** no caso de uma variável só, o R² é literalmente o **quadrado da correlação**
> que vocês calcularam na Aula 3. Correlação e regressão são a mesma ideia vista de ângulos diferentes.

In [ ]:
# Seu código aqui


In [ ]:
# Gráfico: Previsões vs Valores Reais

# Linha perfeita (onde real == previsto)


> **O que esse gráfico mostra?** Se o modelo fosse perfeito, todos os pontos estariam em cima da
> linha vermelha. Quanto mais os pontos se dispersam, mais o modelo erra. Perceba que para diamantes
> mais caros (eixo X > 10.000), as previsões ficam mais espalhadas: a regressão linear tem limitações!

### ⚠️ Um detalhe importante (que vamos resolver na Aula 5)

Repare no que acabamos de fazer: **avaliamos o modelo nos mesmos dados em que ele treinou**.

É como corrigir a prova de um aluno usando exatamente as questões que ele estudou na véspera. A nota
sai boa, mas ela não prova que ele aprendeu de verdade.

Para o objetivo de hoje, que é entender o que é um ajuste linear, isso está de bom tamanho. Mas
guarde a inquietação: **na Aula 5 vamos aprender a avaliar um modelo de forma honesta.**

## 7. Interpretando os Coeficientes

Uma das grandes vantagens da Regressão Linear é que ela é **interpretável**: você consegue entender
o que o modelo aprendeu.

Cada coeficiente representa: *"se essa variável aumentar em 1 unidade (1 quilate, 1 mm, 1 ponto
percentual...), o preço previsto muda em X dólares"*.

In [ ]:
# Extraindo os coeficientes do modelo


In [ ]:
# Visualizando os coeficientes


### Interpretando:

* **Azul (positivo):** quando a variável aumenta, o preço tende a **subir**
* **Vermelho (negativo):** quando a variável aumenta, o preço tende a **cair**

Vamos fazer perguntas sobre o resultado:

* **`carat` tem o maior coeficiente positivo?** Faz sentido! Diamantes maiores custam mais.
* **Alguma variável tem coeficiente negativo?** Pode parecer estranho, mas lembre-se: estamos
  controlando todas as outras variáveis ao mesmo tempo. Uma variável pode ter efeito negativo quando
  combinada com as demais.
* **`x`, `y`, `z` (dimensões físicas)?** Estão correlacionadas com o quilate. Lembra do heatmap lá
  do começo da aula? O modelo pode estar tendo dificuldade de separar o efeito de cada uma.

> **Dica:** coeficientes negativos em variáveis como `depth` e `table` podem indicar que diamantes
> com proporções muito extremas (muito fundos ou muito rasos) valem menos, o que faz sentido na
> joalheria!

> ⚠️ **Cuidado ao comparar coeficientes:** cada variável está em uma unidade diferente (quilates,
> milímetros, porcentagem). Um coeficiente grande pode ser só efeito de a unidade ser pequena. Na
> **Aula 5** vamos aprender uma técnica (*scaling*) que coloca todo mundo na mesma escala justamente
> para tornar essa comparação justa.

## 🎯 Resumo da Aula

| Conceito | Em uma frase |
|---|---|
| **Regressão Linear** | Modelo que usa uma reta para prever valores numéricos |
| **Coeficiente (a)** | Quanto o alvo muda quando a variável sobe 1 unidade |
| **Intercepto (b)** | O valor base, quando todas as variáveis são zero |
| **Ajuste (`fit`)** | Encontrar a reta que minimiza os erros |
| **MAE** | Erro médio em unidade original (ex: $800 de erro médio) |
| **R²** | % da variação explicada pelo modelo |
| **Interpretabilidade** | Conseguir explicar *por que* o modelo previu aquilo |

### O gancho para a próxima aula

Hoje jogamos fora as colunas de texto (`cut`, `color`, `clarity`), usamos variáveis em escalas muito
diferentes e avaliamos o modelo nos próprios dados de treino.

Cada um desses três problemas tem uma solução. É exatamente sobre isso que é a **Aula 5**.

## 🏋️ Mão na Massa

**Exercício 1 (fácil):**

Refaça a regressão simples da seção 3, mas usando `x` (comprimento em mm) no lugar de `carat`.

* Qual é o coeficiente? Como você o traduz em português?
* O R² ficou maior ou menor que o do `carat`?

In [ ]:
# Exercício 1 — Seu código aqui


**Exercício 2 (médio):**

Treine o modelo usando **apenas** `carat`, `depth` e `table` (sem `x`, `y`, `z`).

* O R² caiu muito?
* O coeficiente de `carat` mudou? Por que você acha que isso aconteceu?

In [ ]:
# Exercício 2 — Seu código aqui


**Exercício 3 (difícil, pensamento crítico):**

Olhando o gráfico de "Previsões vs Valores Reais", o modelo erra mais nos diamantes caros. Além
disso, algumas previsões saem **negativas**, e preço negativo não existe!

* Encontre quantas previsões ficaram abaixo de zero.
* Por que uma reta pode prever valores negativos?
* O que isso diz sobre os limites de usar uma reta para descrever esse fenômeno?

In [ ]:
# Exercício 3 — Seu código aqui
